# 17 — Formula Deep Dive Compendium (Advanced Module)

Core formulas from notebooks 10–16 are explained with formal math, symbol-level semantics, assumptions, model-quality impact, research context, and mathematical trade-offs.

## Coverage

- 10: feature-map design, scaling, target encoding  
- 11: polynomial/interaction/aggregation/Shapley formulas  
- 12: empirical risk, CV, bias-variance, nested CV  
- 13: shift, KL/KS, adversarial validation  
- 14: ensembling variance laws and stacking  
- 15: pseudo-labeling semi-supervised objective  
- 16: competition objective, shake, final stack

## Formula 10.1 — Feature-map composition

### Formal definition
$$\hat{y}_i=f_\theta(\phi(x_i)).$$

### Symbols and parameters (word-by-word style)
- $x_i$: raw feature vector.
- $\phi(\cdot)$: engineered map (scaling, interactions, encodings).
- $f_\theta$: model in transformed space.
- $\hat{y}_i$: prediction for row $i$.

### Assumptions
- Same \(\phi\) is used at train and inference time.
- \(\phi\) uses only information available at prediction time (no leakage).

### Why used / effect on model quality (especially post-feature engineering)
- Separates representation learning from model fitting, so feature engineering gains are measurable.
- After strong engineering, simpler models can improve because nonlinearity moved into \(\phi\).

### Research context
- Canonical source: Cover (1965) separability in transformed spaces.
- Empirical takeaway: representation quality often explains more leaderboard lift than swapping base model classes.

### Mathematical advantages
- Modular analysis of approximation vs estimation error.
- Clean ablations: compare feature maps with fixed model.

### Mathematical disadvantages / failure modes
- High-dimensional maps inflate variance/sample complexity.
- Shift-sensitive handcrafted maps can fail under covariate drift.

## Formula 10.2 — Standardization and gradient conditioning

### Formal definition
$$z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j},\quad \mu_j=\frac{1}{n}\sum_i x_{ij},\quad \sigma_j^2=\frac{1}{n}\sum_i(x_{ij}-\mu_j)^2,\quad \frac{\partial \mathcal{L}}{\partial w_j}=\frac{2}{n}\sum_i(\hat{y}_i-y_i)z_{ij}. $$

### Symbols and parameters (word-by-word style)
- $x_{ij}$: raw value of feature $j$ in row $i$.
- $\mu_j,\sigma_j$: train-fold mean and std for feature $j$.
- $z_{ij}$: standardized value.
- $\partial\mathcal{L}/\partial w_j$: coordinate-wise gradient under squared loss.

### Assumptions
- Moments are estimated on training folds only.
- Scale-sensitive optimizers/regularizers are used (linear/NN).

### Why used / effect on model quality (especially post-feature engineering)
- Engineered features (powers, products, counts) often have incompatible scales; normalization balances optimization.
- More isotropic gradients reduce training instability and hyperparameter brittleness.

### Research context
- Canonical source: Efficient BackProp (LeCun et al., 1998) and optimization conditioning literature.
- Empirical takeaway: standardized engineered blocks usually reduce fold variance and improve convergence.

### Mathematical advantages
- Improves conditioning and coordinate fairness under L1/L2 penalties.
- Cheap deterministic transform with clear math.

### Mathematical disadvantages / failure modes
- Mean/std are outlier-sensitive.
- Limited effect for strictly order-based tree splits.

## Formula 10.3 — Smoothed target encoding

### Formal definition
$$\operatorname{TE}(c)=\mathbb{E}[Y\mid C=c],\quad \widehat{\operatorname{TE}}(c)=\frac{n_c\bar{y}_c+\alpha\bar{y}}{n_c+\alpha}. $$

### Symbols and parameters (word-by-word style)
- $c$: category level.
- $n_c$: count for level $c$.
- $\bar{y}_c$: category mean target.
- $\bar{y}$: global target mean.
- $\alpha$: smoothing strength (shrinkage to prior).

### Assumptions
- Encoding is computed OOF/time-safe to prevent leakage.
- Category-target relation is stable enough across folds/test.

### Why used / effect on model quality (especially post-feature engineering)
- Replaces sparse high-cardinality one-hot expansions with dense low-variance signal.
- Shrinkage is crucial after heavy feature engineering where rare categories otherwise overfit.

### Research context
- Canonical source: Micci-Barreca (2001); ordered target stats in CatBoost (Prokhorenkova et al., 2018).
- Empirical takeaway: leakage-safe smoothed TE is often among highest-ROI tabular features.

### Mathematical advantages
- Compact representation for very high-cardinality categories.
- Explicit bias-variance knob via \(\\alpha\).

### Mathematical disadvantages / failure modes
- Leakage risk is severe if computed on full labels.
- Distribution drift in categories can break learned priors.

## Formula 11.1 — Polynomial basis and combinatorial growth

### Formal definition
$$x_1^{\alpha_1}\cdots x_d^{\alpha_d},\ \alpha_j\in\mathbb{N}_0,\ \sum_j\alpha_j\le p,\quad N(d,p)=\binom{d+p}{p}. $$

### Symbols and parameters (word-by-word style)
- $d$: input dimension before expansion.
- $p$: max polynomial degree.
- $\alpha_j$: exponent on feature $j$.
- $N(d,p)$: total engineered monomials up to degree $p$.

### Assumptions
- Target function is smooth enough for low-degree approximation.
- Regularization is present to manage expanded dimensionality.

### Why used / effect on model quality (especially post-feature engineering)
- Adds explicit nonlinearity to linear/meta models.
- The count formula forecasts memory/runtime and overfitting risk before materializing features.

### Research context
- Canonical source: basis expansion treatments in ESL (Hastie-Tibshirani-Friedman).
- Empirical takeaway: degree-2 expansions can help; high degree quickly becomes variance-dominated in tabular data.

### Mathematical advantages
- Deterministic nonlinear expressivity with interpretable terms.
- Closed-form complexity control via \(N(d,p)\).

### Mathematical disadvantages / failure modes
- Combinatorial explosion and multicollinearity.
- Poor extrapolation and numeric instability at high degree.

## Formula 11.2 — Pairwise interactions

### Formal definition
$$\psi_{ij}(x)=x_i x_j.$$

### Symbols and parameters (word-by-word style)
- $x_i,x_j$: base coordinates.
- $\psi_{ij}$: engineered multiplicative interaction.

### Assumptions
- True signal contains non-additive effects.
- Pair list is curated; full \(O(d^2)\) expansion is usually too large.

### Why used / effect on model quality (especially post-feature engineering)
- Captures conditional response patterns missed by additive models.
- After base features are strong, selective interactions often explain residual error.

### Research context
- Canonical source: interaction analysis in nonlinear modeling (e.g., Friedman, 2001 diagnostics).
- Empirical takeaway: targeted interactions can lift CV; indiscriminate expansion usually harms robustness.

### Mathematical advantages
- Simple and interpretable nonlinear term.
- Works across linear/tree/meta models as extra columns.

### Mathematical disadvantages / failure modes
- Feature count and correlation increase rapidly.
- Noise multiplication can reduce generalization.

## Formula 11.3 — Grouped aggregation as conditional expectation

### Formal definition
$$g(x)=\mathbb{E}[Y\mid G(x)].$$

### Symbols and parameters (word-by-word style)
- $G(x)$: grouping key map (entity/time bucket).
- $g(x)$: group-level expected target statistic.

### Assumptions
- Group identifiers are available at inference.
- Smoothing/backoff is used for low-count groups.

### Why used / effect on model quality (especially post-feature engineering)
- Converts sparse entity IDs into dense signal-rich statistics.
- A major post-feature-engineering gain source in user/item/geo/time tabular tasks.

### Research context
- Canonical source: empirical-Bayes shrinkage logic for grouped means; modern target-stat encoders.
- Empirical takeaway: entity aggregates are frequent components of winning Kaggle pipelines.

### Mathematical advantages
- High signal-to-noise when grouping matches latent structure.
- Compact features with strong predictive power.

### Mathematical disadvantages / failure modes
- Temporal leakage risk if future labels enter group stats.
- Rare groups are unstable without shrinkage.

## Formula 11.4 — Shapley value for engineered-feature attribution

### Formal definition
$$\phi_j(v)=\sum_{S\subseteq N\setminus\{j\}}\frac{|S|!(M-|S|-1)!}{M!}\big(v(S\cup\{j\})-v(S)\big).$$

### Symbols and parameters (word-by-word style)
- $N$: feature index set with size $M$.
- $S$: coalition excluding feature $j$.
- $v(S)$: value function of coalition $S$.
- $\phi_j$: attribution assigned to feature $j$.

### Assumptions
- Chosen value function approximates model behavior on relevant background distribution.
- Approximation method is accurate enough for correlated features.

### Why used / effect on model quality (especially post-feature engineering)
- Audits whether engineered features add unique marginal value or only duplicate existing signal.
- Useful for pruning overgrown feature sets with minimal score loss.

### Research context
- Canonical source: Shapley (1953); SHAP framework (Lundberg & Lee, 2017).
- Empirical takeaway: attribution is valuable for debugging, but instability appears with strong collinearity.

### Mathematical advantages
- Satisfies efficiency/symmetry axioms.
- Additive local explanations for complex models.

### Mathematical disadvantages / failure modes
- Exact computation is exponential in feature count.
- Interpretations can be misleading under heavy dependence.

## Formula 12.1 — Empirical risk and K-fold estimator

### Formal definition
$$\hat{R}(f)=\frac{1}{n}\sum_{i=1}^n L(y_i,f(x_i)),\quad \hat{R}_{\mathrm{CV}}(f)=\frac{1}{K}\sum_{k=1}^K\frac{1}{|V_k|}\sum_{i\in V_k}L(y_i,f^{(-k)}(x_i)).$$

### Symbols and parameters (word-by-word style)
- $L$: per-sample loss.
- $\hat{R}(f)$: in-sample empirical risk.
- $V_k$: validation fold.
- $f^{(-k)}$: model fit without fold $k$.
- $\hat{R}_{\mathrm{CV}}$: average out-of-fold risk estimate.

### Assumptions
- Fold strategy matches data geometry (stratified/group/time split as needed).
- All preprocessing/feature engineering is fold-local.

### Why used / effect on model quality (especially post-feature engineering)
- Main bridge from training optimization to expected leaderboard performance.
- Post-feature engineering, CV verifies gains are out-of-sample rather than representation leakage.

### Research context
- Canonical source: Stone (1974) cross-validation consistency line.
- Empirical takeaway: protocol-correct CV correlates with private leaderboard far better than single random split.

### Mathematical advantages
- Data-efficient out-of-sample estimation.
- Produces OOF predictions for stacking and diagnostics.

### Mathematical disadvantages / failure modes
- Compute-heavy with large models/features.
- Still noisy for small data or mismatched split strategy.

## Formula 12.2 — Bias-variance decomposition of CV error

### Formal definition
$$\mathbb{E}[(\hat{R}_{\mathrm{CV}}-R)^2]=\big(\mathbb{E}[\hat{R}_{\mathrm{CV}}]-R\big)^2+\mathrm{Var}(\hat{R}_{\mathrm{CV}}).$$

### Symbols and parameters (word-by-word style)
- $R$: true population risk.
- $\mathbb{E}[\hat{R}_{\mathrm{CV}}]-R$: estimator bias.
- $\mathrm{Var}(\hat{R}_{\mathrm{CV}})$: split/data-induced variance.

### Assumptions
- Expectation is over repeated sample/split draws.
- Second moments are finite.

### Why used / effect on model quality (especially post-feature engineering)
- Explains trade-off: richer feature maps reduce bias but may inflate estimator variance.
- Supports choosing robust models, not only best mean score.

### Research context
- Canonical source: estimator MSE decomposition in statistical decision theory; widely used in model-selection analysis.
- Empirical takeaway: lower-variance candidates often win private LB despite tiny mean-CV deficits.

### Mathematical advantages
- Turns "stability" into a measurable term.
- Improves decision quality when models are close.

### Mathematical disadvantages / failure modes
- True bias to unknown \(R\) is not directly observable.
- Variance estimates from few folds can be noisy.

## Formula 12.3 — Nested CV for unbiased tuning evaluation

### Formal definition
$$\hat{R}_{\text{nested}}=\frac{1}{K_{\text{outer}}}\sum_{k=1}^{K_{\text{outer}}}L\big(y_{V_k},f_{\lambda_k^*}^{(-k)}(x_{V_k})\big),\quad \lambda_k^*=\arg\min_{\lambda\in\Lambda}\hat{R}_{\text{inner}}^{(k)}(\lambda).$$

### Symbols and parameters (word-by-word style)
- $\Lambda$: hyperparameter space.
- $\hat{R}_{\text{inner}}^{(k)}$: inner-loop CV objective.
- $\lambda_k^*$: selected hyperparameter in outer fold $k$.

### Assumptions
- Tuning is fully enclosed in each outer training split.
- Feature engineering choices that use labels are also nested.

### Why used / effect on model quality (especially post-feature engineering)
- Prevents selection bias from reusing validation for tuning and reporting.
- Critical when many engineered-feature variants are tried.

### Research context
- Canonical source: Varma & Simon (2006) on bias from non-nested model selection.
- Empirical takeaway: nested evaluation is slower but more trustworthy in high-search workflows.

### Mathematical advantages
- More honest generalization estimate after tuning.
- Reduces optimistic reporting in leaderboard iteration loops.

### Mathematical disadvantages / failure modes
- High computational cost.
- Small outer folds can increase estimate variance.

## Formula 13.1 — Distribution shift and KL divergence

### Formal definition
$$P_{\text{train}}(X)\neq P_{\text{test}}(X),\quad D_{\mathrm{KL}}(P\parallel Q)=\int p(x)\log\frac{p(x)}{q(x)}\,dx.$$

### Symbols and parameters (word-by-word style)
- $P_{\text{train}},P_{\text{test}}$: train/test feature distributions.
- $D_{\mathrm{KL}}$: information divergence from $Q$ to $P$.
- $p(x),q(x)$: corresponding densities.

### Assumptions
- Density support overlap is sufficient for finite KL.
- Shift diagnostics are computed on representative samples.

### Why used / effect on model quality (especially post-feature engineering)
- Formalizes when CV may stop being predictive of private performance.
- Useful for comparing robustness of alternative feature-engineering pipelines under drift.

### Research context
- Canonical source: Kullback-Leibler (1951); covariate-shift framework (Sugiyama et al., 2007).
- Empirical takeaway: KL-style shift signals are strong warnings but require careful estimation in high dimensions.

### Mathematical advantages
- Information-theoretic, nonnegative mismatch measure.
- Connects to importance weighting and domain adaptation.

### Mathematical disadvantages / failure modes
- Asymmetric and density-estimation sensitive.
- Hard to estimate reliably in large sparse feature spaces.

## Formula 13.2 — KS statistic and adversarial validation AUC

### Formal definition
$$D_{n,m}=\sup_x|F_n(x)-G_m(x)|,\quad \mathrm{AUC}(h(x),s),\ s\in\{0,1\}. $$

### Symbols and parameters (word-by-word style)
- $D_{n,m}$: max CDF gap for two-sample KS test.
- $h(x)$: train-vs-test classifier score.
- $s$: domain label (0 train, 1 test-like).
- $\mathrm{AUC}$: separability of the two domains.

### Assumptions
- KS is applied feature-wise; adversarial model has enough capacity.
- A test-like proxy split exists for domain classification.

### Why used / effect on model quality (especially post-feature engineering)
- KS rapidly screens individual engineered columns; adversarial AUC captures multivariate mismatch.
- Together they identify both where and how shift/leakage likely appears.

### Research context
- Canonical source: Kolmogorov-Smirnov tests; classifier two-sample tests (Lopez-Paz & Oquab, 2017).
- Empirical takeaway: high adversarial AUC often predicts public/private instability.

### Mathematical advantages
- Fast diagnostics with complementary granularity (univariate + multivariate).
- Feature importances from adversarial model are directly actionable.

### Mathematical disadvantages / failure modes
- KS misses dependence structure across features.
- Adversarial AUC depends on classifier choice and tuning.

## Formula 14.1 — Weighted ensembling and variance law

### Formal definition
$$\hat{y}_{ens}=\sum_{m=1}^M w_m\hat{y}^{(m)},\ \sum_m w_m=1,\quad \mathrm{Var}(\hat{y}_{ens})=w^\top\Sigma w,\quad \mathrm{Var}(\bar{y})=\sigma^2\left(\rho+\frac{1-\rho}{M}\right).$$

### Symbols and parameters (word-by-word style)
- $w_m$: blend weight.
- $\Sigma$: covariance matrix of base predictions.
- $\sigma^2$: common base variance in symmetric case.
- $\rho$: pairwise correlation between base models.

### Assumptions
- OOF predictions are used to estimate correlations/covariances.
- Symmetric formula assumes equal variance and equal correlation.

### Why used / effect on model quality (especially post-feature engineering)
- Shows mathematically that diversity (low covariance) matters more than many near-duplicate models.
- After similar feature engineering across models, \(ho\) is often high, limiting ensemble gains.

### Research context
- Canonical source: ensemble theory (Hansen & Salamon, Breiman) and covariance-risk quadratic forms.
- Empirical takeaway: covariance-aware weighting usually outperforms naive equal weighting when models are heterogeneous.

### Mathematical advantages
- Closed-form guidance for weight design and model addition decisions.
- Explains diminishing returns in homogeneous model pools.

### Mathematical disadvantages / failure modes
- Covariance estimates can be noisy on small OOF sets.
- Variance reduction alone does not control ensemble bias.

## Formula 14.2 — Stacking objective with OOF meta-features

### Formal definition
$$\theta^*=\arg\min_\theta\frac{1}{n}\sum_{i=1}^n L\left(y_i,g_\theta\left(\hat{y}^{(1)}_{i,\mathrm{OOF}},\ldots,\hat{y}^{(M)}_{i,\mathrm{OOF}}\right)\right).$$

### Symbols and parameters (word-by-word style)
- $\hat{y}^{(m)}_{i,\mathrm{OOF}}$: leakage-safe base prediction for row $i$.
- $g_\theta$: meta-learner.
- $\theta^*$: optimal meta-parameters.

### Assumptions
- Every meta-feature is generated OOF; no row sees its own target via base fit.
- Base models exhibit complementary error structure.

### Why used / effect on model quality (especially post-feature engineering)
- Learns adaptive, potentially nonlinear combinations beyond fixed weighted averages.
- Frequently extracts extra private-score lift from diverse feature/model pipelines.

### Research context
- Canonical source: Wolpert (1992) stacked generalization.
- Empirical takeaway: stacking helps when diversity is real; otherwise it overfits meta-level noise.

### Mathematical advantages
- Data-driven combiner can improve both accuracy and calibration.
- Naturally absorbs heterogeneous model families.

### Mathematical disadvantages / failure modes
- Leakage-prone if OOF protocol is violated.
- Higher complexity and reproducibility overhead.

## Formula 15.1 — Semi-supervised pseudo-label objective

### Formal definition
$$\mathcal{L}(\theta)=\mathcal{L}_{sup}(\theta)+\lambda\mathcal{L}_{unsup}(\theta),\quad \tilde{y}_u=\mathbf{1}[p_\theta(y=1\mid x_u)\ge\tau],\quad \mathcal{U}_\tau=\{u:\max_c p_\theta(c\mid x_u)\ge\tau\},\quad \mathcal{L}_{unsup}=\frac{1}{|\mathcal{U}_\tau|}\sum_{u\in\mathcal{U}_\tau}\ell(f_\theta(x_u),\tilde{y}_u).$$

### Symbols and parameters (word-by-word style)
- $\lambda$: unsupervised weight.
- $\tau$: confidence threshold.
- $\mathcal{U}_\tau$: accepted unlabeled subset.
- $\tilde{y}_u$: pseudo-label.
- $\mathcal{L}_{unsup}$: loss on accepted unlabeled rows.

### Assumptions
- Model confidence is informative (calibrated enough).
- Unlabeled pool is distributionally related to target data.

### Why used / effect on model quality (especially post-feature engineering)
- Adds training signal when labeled data are limited and feature engineering has saturated supervised gains.
- Thresholding controls error propagation from noisy pseudo-labels.

### Research context
- Canonical source: Pseudo-Label (Lee, 2013); confidence-threshold SSL refinements such as FixMatch (Sohn et al., 2020).
- Empirical takeaway: gains are real but fragile; confidence filtering and schedule tuning are decisive.

### Mathematical advantages
- Simple extension to existing supervised pipelines.
- Can improve sample efficiency and decision boundaries.

### Mathematical disadvantages / failure modes
- Confirmation bias if pseudo-labels are wrong but confident.
- Class imbalance can worsen if thresholding favors majority classes.

## Formula 16.1 — Competition objective and leaderboard shake

### Formal definition
$$\theta^*=\arg\min_{\theta\in\Theta}\mathbb{E}[\mathcal{M}_{private}(\theta)],\quad \widehat{\mathcal{M}}_{public}(\theta)=\mathcal{M}(\theta;\mathcal{D}_{public}),\quad \widehat{\mathcal{M}}_{private}(\theta)=\mathcal{M}(\theta;\mathcal{D}_{private}),\quad \Delta_{shake}(\theta)=\widehat{\mathcal{M}}_{private}(\theta)-\widehat{\mathcal{M}}_{public}(\theta).$$

### Symbols and parameters (word-by-word style)
- $\Theta$: candidate model/feature/ensemble configurations.
- $\mathcal{D}_{public},\mathcal{D}_{private}$: leaderboard-visible and hidden splits.
- $\Delta_{shake}$: public-to-private score shift.

### Assumptions
- Public feedback is a noisy proxy for private performance.
- Submission pipeline is identical on both hidden subsets.

### Why used / effect on model quality (especially post-feature engineering)
- Mathematically frames competition strategy as decision under uncertainty, not pure public-score maximization.
- Encourages selecting stable pipelines with lower expected private regret after feature engineering and ensembling.

### Research context
- Canonical source: adaptive holdout/leaderboard overfitting analyses (e.g., Blum & Hardt, 2015).
- Empirical takeaway: models tuned too aggressively on public feedback often degrade on private leaderboard.

### Mathematical advantages
- Unifies CV stability, ensemble diversity, and leaderboard risk in one objective view.
- Supports simulation-based robustness checks.

### Mathematical disadvantages / failure modes
- True private expectation is unobservable during competition.
- Estimated shake can be noisy when validation sample is small.

## Formula 16.2 — Stacked competition submission function

### Formal definition
$$\hat{y}=g_\phi\big(\hat{y}^{(torch)},\hat{y}^{(xgb)},\hat{y}^{(lgbm)},\hat{y}^{(rf)}\big).$$

### Symbols and parameters (word-by-word style)
- $\hat{y}^{(\cdot)}$: base predictions from heterogeneous model families.
- $g_\phi$: meta-combiner with parameters $\phi$.
- $\hat{y}$: final submission prediction.

### Assumptions
- Base models are trained with aligned folds and leakage-safe preprocessing.
- Meta-combiner is fit on OOF base predictions.

### Why used / effect on model quality (especially post-feature engineering)
- Combines distinct inductive biases that interact differently with engineered features.
- Often improves robustness versus single-family ensembles in final submission stage.

### Research context
- Canonical source: stacked generalization (Wolpert, 1992) and practical stacked ensembles in large-scale competitions.
- Empirical takeaway: heterogeneous stacks are common in top solutions when protocol discipline is strict.

### Mathematical advantages
- Can increase both score and robustness to split noise.
- Flexible architecture for incremental model additions.

### Mathematical disadvantages / failure modes
- Pipeline orchestration complexity is high.
- Gains may be small if base predictions are too correlated.